<a href="https://colab.research.google.com/github/sofia-seo-j/chantey_2026/blob/BDS/BDS_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Cousre: Big Data Statistics*

S.E. Jeong (2894452)

# Assignment Part I

## Load

In [38]:
# Packages inmported
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [39]:
# Data immported
data = pd.read_csv('/content/Assignment_BDS_25_26_data.csv')
print("\n Summary of the data:")
data.info()



 Summary of the data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   radius_mean              540 non-null    float64
 1   texture_mean             540 non-null    float64
 2   perimeter_mean           540 non-null    float64
 3   area_mean                540 non-null    float64
 4   smoothness_mean          540 non-null    float64
 5   compactness_mean         540 non-null    float64
 6   concavity_mean           540 non-null    float64
 7   concave.points_mean      540 non-null    float64
 8   symmetry_mean            540 non-null    float64
 9   fractal_dimension_mean   540 non-null    float64
 10  radius_se                540 non-null    float64
 11  texture_se               540 non-null    float64
 12  perimeter_se             540 non-null    float64
 13  area_se                  540 non-null    float64
 14  smo

## Task 1

In [40]:
# Linear model with all the explanatory variables
X = data.drop(columns=["radius_mean"])
X = sm.add_constant(X)
y = data["radius_mean"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            radius_mean   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 6.454e+04
Date:                Sat, 14 Feb 2026   Prob (F-statistic):               0.00
Time:                        20:24:23   Log-Likelihood:                 772.38
No. Observations:                 540   AIC:                            -1485.
Df Residuals:                     510   BIC:                            -1356.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [41]:
# t-test using the results
alpha = 0.05
pvals = model.pvalues
significant = pvals[pvals < alpha]

print("\n Significant explanatory variables (5% level) \n")
for var, p in significant.items():
    print(f"{var}: p-value = {round(p, 4)}")


 Significant explanatory variables (5% level) 

const: p-value = 0.0027
perimeter_mean: p-value = 0.0
area_mean: p-value = 0.0
smoothness_mean: p-value = 0.0002
compactness_mean: p-value = 0.0
concavity_mean: p-value = 0.0
perimeter_se: p-value = 0.0
concavity_se: p-value = 0.0002
concave.points_se: p-value = 0.0123
radius_worst: p-value = 0.0
perimeter_worst: p-value = 0.0
area_worst: p-value = 0.0
smoothness_worst: p-value = 0.0026
compactness_worst: p-value = 0.0005


## Task 2

In [42]:
# Variable backward elimination
threshold = 3.86
while True:

    full_model = sm.OLS(y, X).fit()
    SSE_full = sum(full_model.resid**2)

    n = full_model.nobs
    d = len(full_model.params)

    F_stats = {}

    for var in X.columns:

        X_restricted = X.drop(columns=[var])
        restricted_model = sm.OLS(y, X_restricted).fit()
        SSE_restricted = sum(restricted_model.resid**2)

        F = (SSE_restricted - SSE_full) / (SSE_full / (n - d))
        F_stats[var] = F

    worst_var = min(F_stats, key=F_stats.get)
    min_F = F_stats[worst_var]

    if min_F > threshold:
        break

    print(f"Removing {worst_var} (F = {min_F:.2f})")
    X = X.drop(columns=[worst_var])

Removing concave.points_worst (F = 0.00)
Removing texture_mean (F = 0.01)
Removing symmetry_se (F = 0.04)
Removing concave.points_mean (F = 0.11)
Removing concavity_worst (F = 0.14)
Removing compactness_se (F = 0.30)
Removing fractal_dimension_worst (F = 0.34)
Removing symmetry_worst (F = 0.97)
Removing area_se (F = 0.94)
Removing smoothness_se (F = 1.04)
Removing symmetry_mean (F = 1.81)
Removing texture_se (F = 2.53)
Removing texture_worst (F = 1.28)
Removing fractal_dimension_mean (F = 3.53)
Removing fractal_dimension_se (F = 1.85)


In [43]:
model2 = sm.OLS(y, X).fit()

alpha = 0.05
pvals2 = model2.pvalues

# Keep only significant ones
significant2 = pvals2[pvals2 < alpha]

print("\nSignificant explanatory variables after backward elimination (5% level)\n")

for var, p in significant2.items():
    print(f"{var}: p-value = {round(p, 4)}")


Significant explanatory variables after backward elimination (5% level)

const: p-value = 0.0
perimeter_mean: p-value = 0.0
area_mean: p-value = 0.0
smoothness_mean: p-value = 0.0
compactness_mean: p-value = 0.0
concavity_mean: p-value = 0.0
radius_se: p-value = 0.0143
perimeter_se: p-value = 0.0
concavity_se: p-value = 0.0
concave.points_se: p-value = 0.0
radius_worst: p-value = 0.0
perimeter_worst: p-value = 0.0
area_worst: p-value = 0.0
smoothness_worst: p-value = 0.0
compactness_worst: p-value = 0.0


In [44]:
# Task 1 significant variables
task1_vars = set(significant.index)

# Task 2 significant variables
task2_vars = set(significant2.index)

print("\nVariables in Task 1 but not Task 2:")
print(task1_vars - task2_vars)

print("\nVariables in Task 2 but not Task 1:")
print(task2_vars - task1_vars)


Variables in Task 1 but not Task 2:
set()

Variables in Task 2 but not Task 1:
{'radius_se'}
